<a href="https://colab.research.google.com/github/JasmineLee7/movie-rec/blob/main/notebooks/03_low_rank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 3!

# Part A

In [9]:
import os
import subprocess
import sys

USER = 'JasmineLee7'
REPO = 'movie-rec'

if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone', f'https://github.com/{USER}/{REPO}.git', REPO],
        check=True)

sys.path.insert(0, f'{REPO}/src')
from evaluate import split_ratings, to_matrix, rmse
from models import bias_baseline

This code connects to GitHub to clone or update the project repository. It then updates the system path so Python can locate and import the custom evaluation and baseline functions saved during Milestone 2.

In [10]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if not os.path.exists('ml-100k'):
    urllib.request.urlretrieve(
        'https://files.grouplens.org/datasets/movielens/ml-100k.zip',
        'ml-100k.zip')
    zipfile.ZipFile('ml-100k.zip').extractall()

ratings = pd.read_csv(
    'ml-100k/u.data', sep='\t',
    names=['user', 'item', 'rating', 'time'])

train, test = split_ratings(ratings, seed=0)
fit, val = split_ratings(train, seed=1)

print("fit:", len(fit), "validation:", len(val), "test:", len(test))

fit: 81200 validation: 8832 test: 9968


This code downloads and extracts the MovieLens 100k dataset if it is not already present. It reads the ratings into a Pandas DataFrame and splits the data twice using a fixed seed: first reserving a hidden test set for final evaluation, and then splitting the rest into a fitting set (81,200 ratings) and a validation set (8,832 ratings) for tuning the models.

## Part B: Set up a fair validation comparison

In [11]:
R_fit = to_matrix(fit)
known_fit = ~np.isnan(R_fit)
val_truth = val["rating"].to_numpy()
val_u = val["user"].to_numpy() - 1
val_i = val["item"].to_numpy() - 1

def score_at(P, rows, cols, truth):
    """RMSE for selected entries of a prediction matrix."""
    pred = np.clip(P[rows, cols], 1, 5)
    return rmse(pred, truth)

This code creates a user-by-movie matrix using only the fitting ratings. `known_fit` marks entries that contain an observed fitting rating. Validation and test ratings stay out of this matrix so they cannot influence the models. `val_truth` stores the actual validation ratings, and `val_u` and `val_i` store the matching user and movie positions. We subtract 1 because MovieLens IDs start at 1, while Python positions start at 0. The `score_at` function selects the predictions for those user–movie pairs, clips them to the 1–5 star range, and compares them with the actual ratings using RMSE. A lower validation RMSE means more accurate predictions on these held-out ratings; later sections use this score to choose model settings.

## Part C: Build the baseline prediction matrix

In [12]:

def baseline_matrix(R):
    """Full matrix: overall mean plus user and movie effects."""
    mu, b_user, b_item = bias_baseline(R)
    return mu + b_user[:, None] + b_item[None, :]

B_fit = baseline_matrix(R_fit)
baseline_val_rmse = score_at(B_fit, val_u, val_i, val_truth)
print("Baseline validation RMSE:", round(baseline_val_rmse, 4))


Baseline validation RMSE: 0.9731


This code creates a baseline prediction matrix using the average rating and adjustments for each user and movie. It then checks how accurate the predictions are by calculating the RMSE using the validation data. This gives us a starting point to see if the SVD models in the next sections can make better predictions.

# Part D: Create a rank-k helper

In [13]:
def rank_k(U, s, Vt, k):
# Build a rank-k matrix from an already computed SVD.
  return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

We define a helper function that uses the first *k* singular values and their matching singular vectors to reconstruct a low-rank approximation of a matrix. This lets us compare different ranks without recomputing the SVD.

# Part E: Model 1 — fill gaps with movie averages

In [14]:
mu_fit = np.nanmean(R_fit)
n_item = known_fit.sum(axis=0)
item_mean = np.full(R_fit.shape[1], mu_fit)
has_rating = n_item > 0
item_mean[has_rating] = (
  np.nansum(R_fit, axis=0)[has_rating] / n_item[has_rating])

F = np.where(known_fit, R_fit, item_mean[None, :])
U1, s1, Vt1 = np.linalg.svd(F, full_matrices=False)

Fills missing ratings with each movie’s average from the fitting data, using the overall average if a movie has no fitting ratings. It then computes the SVD of the completed matrix, preparing Model 1 to make predictions using low-rank approximations.

# Part F: Model 2 — let SVD explain what the baseline misses

In [15]:
X_fit = np.where(known_fit, R_fit - B_fit, 0.0)
U2, s2, Vt2 = np.linalg.svd(X_fit, full_matrices=False)

Subtracts the baseline predictions from the known fitting ratings and fills missing deviations with zero. It then computes the SVD of this residual matrix to capture patterns the baseline misses. Later, these low-rank deviations are added back to the baseline to predict ratings.

## Part H: Model 3 — Iteratively Improve the Missing-Value Guesses

In [16]:

def iterative_svd_predict(R, k, n_iter=10):
    """Predict ratings by repeatedly imputing missing deviations."""
    known = ~np.isnan(R)
    B = baseline_matrix(R)
    X = np.where(known, R - B, 0.0)

    for _ in range(n_iter):
        U, s, Vt = np.linalg.svd(X, full_matrices=False)
        L = rank_k(U, s, Vt, k)
        X = np.where(known, R - B, L)

    return L + B


Instead of leaving the missing ratings as zero, this function tries to find better estimates for them. It uses the baseline as a starting point and runs SVD multiple times to adjust the missing values. The ratings we already know stay the same throughout the process. At the end the function returns a matrix of predicted ratings.

In [17]:

ks_iter = [2, 5, 10, 20]
val_err3 = []

for k in ks_iter:
    P3 = iterative_svd_predict(R_fit, k, n_iter=10)
    val_err3.append(score_at(P3, val_u, val_i, val_truth))

best_k3 = ks_iter[int(np.argmin(val_err3))]
print("Best iterative rank:", best_k3)
print("Best validation RMSE:", min(val_err3))


Best iterative rank: 2
Best validation RMSE: 0.9431464026691649


We compared ranks 2, 5, 10, and 20 using 10 iterations for each. Rank 2 ended up performing the best with a validation RMSE of 0.9431. This was lower than the baseline RMSE of 0.9731, showing that the iterative approach improved the predictions.

# Part J: Make a Few Recommendations


In [18]:
item_info = pd.read_csv(
    "ml-100k/u.item", sep="|", header=None, usecols=[0, 1],
    encoding="latin-1"
)
item_info.columns = ["item", "title"]

user_id = 1
user_row = user_id - 1
unrated = np.where(np.isnan(R_train[user_row]))[0]
predicted_scores = np.clip(test_prediction[user_row], 1, 5)
top = unrated[np.argsort(predicted_scores[unrated])[-5:][::-1]]

for col in top:
    title = item_info.loc[item_info["item"] == col + 1, "title"].iloc[0]
    print(title, "predicted rating:", round(predicted_scores[col], 2))

NameError: name 'R_train' is not defined

This section generates top movie recommendations for a specific user using the chosen prediction matrix. It loads the movie titles from `u.item`, identifies all the movies that user 1 has not yet rated in the training data, retrieves their predicted star ratings (clipped to the 1-5 range), and prints out the titles and predicted scores for the top 5 highest-scoring recommendations.